# Almgren–Chriss — companion notebook

Companion for [`almgren-chriss.md`](./almgren-chriss.md). Inventory trajectories at several risk-aversion levels and the execution efficient frontier.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Parameters (continuous-time form)
X = 1_000_000  # shares to liquidate
T = 1.0        # horizon (1 trading day)
eta = 2.5e-6   # temporary impact per unit rate (price$ per share per (share/time))
sigma = 0.02   # vol

def trajectory(lam, n=200):
    t = np.linspace(0, T, n)
    if lam == 0:
        return t, X*(1 - t/T)              # linear / TWAP
    kappa = np.sqrt(lam * sigma**2 / eta)
    return t, X * np.sinh(kappa*(T - t)) / np.sinh(kappa*T)

def expected_cost(lam, n=200):
    t, x = trajectory(lam, n)
    dt = t[1] - t[0]
    v = -np.gradient(x, dt)
    return eta * np.sum(v**2) * dt

def variance_cost(lam, n=200):
    t, x = trajectory(lam, n)
    dt = t[1] - t[0]
    return sigma**2 * np.sum(x**2) * dt

lams = [0, 1e-7, 1e-6, 1e-5, 1e-4]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for lam in lams:
    t, x = trajectory(lam)
    axes[0].plot(t, x, label=f'lambda={lam:.0e}')
axes[0].set_xlabel('time'); axes[0].set_ylabel('inventory x(t)')
axes[0].set_title('Optimal inventory trajectories'); axes[0].legend()

# Efficient frontier
lam_grid = np.logspace(-9, -3, 80)
costs = [expected_cost(l) for l in lam_grid]
vars_ = [variance_cost(l) for l in lam_grid]
axes[1].plot(np.sqrt(vars_), costs, '.-')
axes[1].set_xlabel('sqrt(variance)  (cost uncertainty)'); axes[1].set_ylabel('expected impact cost')
axes[1].set_title('Execution efficient frontier')
plt.tight_layout(); plt.show()